In [12]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image

from src import glob_vars

ROOT_DIR = glob_vars['root_dir']

# ==========================================
# 1. LOAD MODEL
# ==========================================
# Ganti dengan path file model .keras kamu
MODEL_PATH = "asr_s5.keras"
model = load_model(MODEL_PATH)
print("Model berhasil di-load!")

# ==========================================
# 2. PREPROCESS GAMBAR TEST
# ==========================================
# Ganti dengan path gambar yang mau ditest
IMAGE_PATH = f"{ROOT_DIR}/ASD Data/ASD Data/Test/autism/001.jpg"

# Tentukan target size sesuai input_shape pas training (misal: 224x224)
TARGET_SIZE = (224, 224)

# Load gambar dan ubah ukurannya
img = image.load_img(IMAGE_PATH, target_size=TARGET_SIZE)

# Konversi gambar ke numpy array
img_array = image.img_to_array(img)

# Normalisasi pixel (0-255 jadi 0.0-1.0)
# Catatan: Sesuaikan ini dengan pra-pemrosesan pas kamu training!
img_array = img_array / 255.0

# Tambahkan dimensi batch: dari (224, 224, 3) jadi (1, 224, 224, 3)
img_array = np.expand_dims(img_array, axis=0)

# ==========================================
# 3. PREDIKSI
# ==========================================
predictions = model.predict(img_array)

# ==========================================
# 4. INTERPRETASI HASIL
# ==========================================
# Contoh A: Klasifikasi Biner (1 Node Output dengan Activation Sigmoid / BCE)
confidence = predictions[0][0]
if confidence >= 0.5:
    label = "Non-Autism (1)"
    score = confidence * 100
else:
    label = "Autism (0)"
    score = (1 - confidence) * 100

print(f"\nHasil Prediksi : {label}")
print(f"Tingkat Keyakinan (Confidence): {score:.2f}%")
print(f"Raw Output Probability        : {confidence:.4f}")

Model berhasil di-load!
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step

Hasil Prediksi : Autism (0)
Tingkat Keyakinan (Confidence): 99.25%
Raw Output Probability        : 0.0075


In [9]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array

# 1. Path ke file model & folder test
MODEL_PATH = 'asr_s5.keras'  
TEST_DIR = f'{ROOT_DIR}/ASD Data/ASD Data/Test'  # Ganti dengan path folder test kamu
IMG_SIZE = (224, 224)

# 2. Load model yang sudah disave
print("Loading model...")
model = load_model(MODEL_PATH)
print("Model berhasil di-load!")

# 3. Label kelas (pastikan urutan alfabetis sesuai dengan folder Keras ImageDataGenerator)
class_names = ['autistic', 'non-autistic']

# ==========================================
# OPSI A: Prediksi 1 Folder Data Test (Batch)
# ==========================================
test_datagen = ImageDataGenerator(rescale=1./255) # Sesuaikan preprocessing rescaling-nya

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=32,
    class_mode='binary', # atau 'categorical' tergantung setup awal
    shuffle=False
)

# Ambil 1 batch gambar
test_images, test_labels = next(test_generator)

# Prediksi batch
predictions = model.predict(test_images)

# Visualisasi 12 Sampel
plt.figure(figsize=(15, 10))

for i in range(min(12, len(test_images))):
    plt.subplot(3, 4, i + 1)
    
    img = test_images[i]
    prob = predictions[i][0]
    
    # Ambil index prediksi (Binary Threshold = 0.5)
    pred_idx = 1 if prob > 0.5 else 0
    actual_idx = int(test_labels[i]) if test_labels.ndim == 1 else int(np.argmax(test_labels[i]))
    
    pred_label = class_names[pred_idx]
    actual_label = class_names[actual_idx]
    
    # Confidence Score (%)
    confidence = prob if pred_idx == 1 else (1 - prob)
    
    # Warna Teks: Hijau (Benar), Merah (Salah)
    color = 'green' if pred_idx == actual_idx else 'red'
    
    plt.imshow(img)
    plt.title(f"Pred: {pred_label} ({confidence*100:.1f}%)\nActual: {actual_label}", 
              color=color, fontsize=10, fontweight='bold')
    plt.axis('off')

plt.tight_layout()
plt.show()

Loading model...
Model berhasil di-load!
Found 300 images belonging to 2 classes.


UnidentifiedImageError: cannot identify image file <_io.BytesIO object at 0x7fe55ce60180>